# TimeR4 + TEN v7

**Fix trong v7:**
- ✅ Fix version conflict `peft` vs `huggingface_hub`
- ✅ Chỉ 4 pattern cụ thể (ngày/tháng có số rõ ràng)
- ✅ Assert kiểm tra coverage < 80% trước khi chạy pipeline

> **Kaggle**: GPU T4 · Internet ON · Persistence = Files only · **Restart Session trước khi Run All**

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 1 — FIX VERSION + CÀI THƯ VIỆN + LOAD MODEL
# ═══════════════════════════════════════════════════════════

import subprocess, sys

# Bước 1: Gỡ TẤT CẢ thư viện liên quan để tránh conflict
print('Gỡ thư viện cũ...')
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y',
    'peft', 'sentence-transformers', 'transformers',
    'huggingface-hub', 'accelerate', 'triton'],
    capture_output=True)

# Bước 2: Cài đúng version tương thích
pkgs = [
    'huggingface_hub==0.23.4',
    'peft==0.11.1',
    'transformers==4.44.0',
    'sentence-transformers==3.3.1',
    'accelerate==0.34.0',
    'faiss-cpu',
    'tqdm',
    'spacy',
]
for pkg in pkgs:
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg],
                       capture_output=True, text=True)
    print(f'  {"✅" if r.returncode==0 else "❌"} {pkg}')

subprocess.run([sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm'],
               capture_output=True)
print('  ✅ en_core_web_sm')

# Bước 3: Import tất cả
import torch, os, json, glob, sys, re, copy
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
print('\n✅ Import OK')

# Bước 4: Load Mistral
MODEL_NAME = 'mistralai/Mistral-7B-Instruct-v0.2'
print(f'\nLoading {MODEL_NAME}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
llm = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16, device_map='auto'
)
vram = torch.cuda.memory_allocated()/1e9 if torch.cuda.is_available() else 0
print(f'✅ Mistral loaded! VRAM: {vram:.1f} GB')

# Bước 5: Retriever
os.makedirs('/kaggle/working/models', exist_ok=True)
RETRIEVER_PATH = '/kaggle/working/models/multi_model'
if not os.path.exists(RETRIEVER_PATH):
    SentenceTransformer('all-MiniLM-L6-v2').save(RETRIEVER_PATH)
print(f'✅ Retriever: {RETRIEVER_PATH}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 2 — MODULE TEN v7: CHỈ 4 PATTERN CỤ THỂ
# KHÔNG dùng TEN_PATTERNS global — định nghĩa cứng trong hàm
# ═══════════════════════════════════════════════════════════

def normalize_temporal_expression(question):
    """
    TEN v7: Chỉ normalize 4 pattern CỰC KỲ CỤ THỂ.
    Tất cả đều yêu cầu có SỐ rõ ràng — không bao giờ bắt nhầm.

    Pattern 1: ngày DD tháng MM năm YYYY  →  YYYY-MM-DD
    Pattern 2: ngày DD/MM/YYYY            →  YYYY-MM-DD
    Pattern 3: tháng MM năm YYYY          →  YYYY-MM
    Pattern 4: tháng MM/YYYY             →  YYYY-MM
    """
    text = question.lower()
    spans = []  # (start, end, replacement)

    # Pattern 1: ngày DD tháng MM năm YYYY
    for m in re.finditer(
            r'ngày\s+(\d{1,2})\s+tháng\s+(\d{1,2})\s+năm\s+(\d{4})',
            text):
        rep = f'{m.group(3)}-{int(m.group(2)):02d}-{int(m.group(1)):02d}'
        spans.append((m.start(), m.end(), rep))

    # Pattern 2: ngày DD/MM/YYYY
    for m in re.finditer(
            r'ngày\s+(\d{1,2})/(\d{1,2})/(\d{4})',
            text):
        rep = f'{m.group(3)}-{int(m.group(2)):02d}-{int(m.group(1)):02d}'
        spans.append((m.start(), m.end(), rep))

    # Pattern 3: tháng MM năm YYYY
    for m in re.finditer(
            r'tháng\s+(\d{1,2})\s+năm\s+(\d{4})',
            text):
        rep = f'{m.group(2)}-{int(m.group(1)):02d}'
        spans.append((m.start(), m.end(), rep))

    # Pattern 4: tháng MM/YYYY
    for m in re.finditer(
            r'tháng\s+(\d{1,2})/(\d{4})',
            text):
        rep = f'{m.group(2)}-{int(m.group(1)):02d}'
        spans.append((m.start(), m.end(), rep))

    if not spans:
        return text

    # Loại overlap — giữ match dài nhất
    kept, last_end = [], -1
    for s, e, r in sorted(spans, key=lambda x: -(x[1]-x[0])):
        if s >= last_end:
            kept.append((s, e, r))
            last_end = e
    kept.sort(key=lambda x: x[0])
    for s, e, r in reversed(kept):
        text = text[:s] + r + text[e:]
    return text


# ── Test bắt buộc ────────────────────────────────────────
TESTS = [
    # (phải_normalize, câu_hỏi)
    (True,  'Trước ngày 22 tháng 10 năm 2008, Malaysia đã làm gì?'),
    (True,  'Ai đến thăm Kuwait sau ngày 2 tháng 2 năm 2015?'),
    (True,  'Vào ngày 12 tháng 4 năm 2008, nước nào buộc tội Ethiopia?'),
    (True,  'Ai ký thỏa thuận với TQ vào tháng 4 năm 2005?'),
    (True,  'Nước nào lên án TQ vào tháng 8/2013?'),
    (True,  'Ai đến thăm Iraq vào ngày 14/10/2015?'),
    (False, 'Sau Bộ Quốc phòng Đan Mạch, ai đến Iraq đầu tiên?'),
    (False, 'Lần cuối cùng Đài Loan yêu cầu TQ là năm nào?'),
    (False, 'Trong những năm gần đây ai lãnh đạo Anh?'),
    (False, 'Sau Hội đồng Bộ trưởng Kazakhstan, ai đàm phán với TQ?'),
]

print('=== TEST MODULE TEN v7 ===')
ok = 0
for should_change, q in TESTS:
    result  = normalize_temporal_expression(q)
    changed = result != q.lower()
    correct = (changed == should_change)
    if correct: ok += 1
    icon = '✅' if correct else '❌'
    tag  = 'normalize' if should_change else 'giữ nguyên'
    print(f'{icon} [{tag}]: {q[:55]}')
    if changed:
        print(f'   → {result[:55]}')

print(f'\nKết quả: {ok}/{len(TESTS)}')
assert ok == len(TESTS), f'❌ TEN test thất bại ({ok}/{len(TESTS)}) — DỪNG LẠI!'
print('✅ Module TEN v7 hoạt động đúng — tiếp tục Cell 3')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 3 — CLONE TIMER4 + PATCH + LOAD DATA
# ═══════════════════════════════════════════════════════════

if not os.path.exists('/kaggle/working/TimeR4'):
    os.system('git clone --depth 1 https://github.com/qianxinying/TimeR4.git /kaggle/working/TimeR4')
os.chdir('/kaggle/working/TimeR4')

with open('retrival.py') as f: code = f.read()
patches = [
    ('self.triplet_id_list = [[triple[0], triple[1], triple[2], triple[3]] for triple in id_list]',
     'self.triplet_id_list = [[triple[0], triple[1], triple[2], triple[3]] for triple in id_list] if id_list is not None else []'),
    ('self.model = SentenceTransformer(model_name, device="cuda")\n        self.embedding_size = embedding_size',
     'self.model = SentenceTransformer(model_name, device="cuda")\n        self.embedding_size = self.model.get_sentence_embedding_dimension()'),
    ('self.full_time = [datetime.strptime(triple[3], "%Y-%m-%d").date() for triple in triple_list]',
     'self.full_time = []\n        for triple in triple_list:\n            t = triple[3] if len(triple) > 3 else "2000-01-01"\n            if len(t) == 4: t += "-01-01"\n            elif len(t) == 7: t += "-01"\n            try: self.full_time.append(datetime.strptime(t[:10], "%Y-%m-%d").date())\n            except: self.full_time.append(datetime.strptime("2000-01-01", "%Y-%m-%d").date())'),
    ('corpus_embeddings = corpus_embeddings / np.linalg.norm(corpus_embeddings, axis=1)[:, None]',
     'if corpus_embeddings.ndim == 1:\n            corpus_embeddings = corpus_embeddings[np.newaxis, :]\n        corpus_embeddings = corpus_embeddings / np.linalg.norm(corpus_embeddings, axis=1)[:, None]'),
]
for old, new in patches:
    if old in code: code = code.replace(old, new)
with open('retrival.py', 'w') as f: f.write(code)
if 'retrival' in sys.modules: del sys.modules['retrival']
from retrival import Retrieval
print('✅ retrival.py patched + loaded')

# Load dataset
found = glob.glob('/kaggle/input/**/MultiTQ_vi_test.json', recursive=True)
DATASET_PATH = found[0] if found else 'datasets/MultiTQ/questions/test.json'
print(f'✅ Dataset: {DATASET_PATH}')

with open(DATASET_PATH, encoding='utf-8') as f:
    all_q = json.load(f)

def get_text(item): return item.get('question', item.get('Question', ''))

# Lọc câu CÓ temporal expression
temporal_q = [q for q in all_q
              if normalize_temporal_expression(get_text(q)) != get_text(q).lower()]

pct = len(temporal_q)/len(all_q)*100
print(f'\nTổng: {len(all_q):,} câu')
print(f'CÓ temporal: {len(temporal_q):,} ({pct:.1f}%)')

# Bảo vệ: nếu coverage > 80% → TEN vẫn sai, không chạy tiếp
assert pct < 80, f'❌ Coverage {pct:.1f}% > 80% — TEN vẫn bắt nhầm! DỪNG LẠI.'
print(f'✅ Coverage hợp lý ({pct:.1f}%) — tiếp tục')

N_TOTAL   = 1000
questions = temporal_q[:N_TOTAL]
print(f'Dùng {len(questions)} câu')

print('\nVí dụ 3 câu:')
for q in questions[:3]:
    orig = get_text(q)
    norm = normalize_temporal_expression(orig)
    print(f'  Gốc: {orig}')
    print(f'  →    {norm}\n')

# Load TKG
triple_list = []
with open('datasets/MultiTQ/kg/full.txt', encoding='utf-8') as f:
    for line in f:
        line = line.strip().replace('_', ' ')
        if not line: continue
        parts = line.split('\t') if '\t' in line else line.split()
        if len(parts) >= 3: triple_list.append(parts)
print(f'✅ TKG: {len(triple_list):,} triplets')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 4 — HÀM PIPELINE
# ═══════════════════════════════════════════════════════════

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

def call_llm(prompt, max_new_tokens=80):
    inp = tokenizer(prompt, return_tensors='pt', truncation=True,
                    max_length=512).to(DEVICE)
    with torch.no_grad():
        out = llm.generate(**inp, max_new_tokens=max_new_tokens,
                           do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inp['input_ids'].shape[1]:],
                             skip_special_tokens=True).strip()

def gen_answer(prompt, max_new_tokens=64):
    inp = tokenizer(prompt, return_tensors='pt', truncation=True,
                    max_length=1024).to(DEVICE)
    with torch.no_grad():
        out = llm.generate(**inp, max_new_tokens=max_new_tokens,
                           do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inp['input_ids'].shape[1]:],
                             skip_special_tokens=True).strip().split('\n')[0]

def hit(pred, gold):
    p = str(pred).lower().strip()
    gs = gold if isinstance(gold, list) else [gold]
    return any(str(g).lower().strip() in p or p in str(g).lower().strip()
               for g in gs)

def get_q(x):    return x.get('question', x.get('Question', ''))
def get_gold(x): return x.get('answer', x.get('answers',
                       x.get('Answer', x.get('answers_simple', '?'))))

def run_pipeline(qs_raw, triples, use_ten=False, label=''):
    qs = copy.deepcopy(qs_raw)
    print(f'\n{"="*58}')
    print(f'  {label}  TEN={"ON" if use_ten else "OFF"}  n={len(qs)}')
    print(f'{"="*58}')

    q_list = [get_q(q) for q in qs]
    q_norm = [normalize_temporal_expression(q) for q in q_list] if use_ten else q_list

    if use_ten:
        n = sum(1 for a,b in zip(q_list,q_norm) if a.lower()!=b)
        pct = n/len(q_list)*100
        print(f'[TEN] Normalized: {n}/{len(q_list)} ({pct:.1f}%)')
        if pct > 80:
            print('⚠️  WARNING: >80% — TEN có thể vẫn đang bắt nhầm!')

    print('[1/4] Retrieve background...')
    r1 = Retrieval(RETRIEVER_PATH, q_norm, triples, None, None)
    d1,c1 = r1.compute_similarity(n=15)
    bg = r1.get_result(d1, c1, q_norm, re_rank=False)

    print('[2/4] Rewrite...')
    for i in tqdm(range(len(qs)), desc='Rewrite', leave=False):
        q = get_q(qs[i])
        f = (bg[i].get('fact') or [''])[0]
        resp = call_llm(
            f'Rewrite with explicit time. Output question only.\n'
            f'Fact: {f}\nQuestion: {q}\nRewritten:'
        ).split('\n')[0].strip()
        qs[i]['question'] = resp or q

    print('[3/4] Retrieve + Rerank...')
    q2 = [get_q(q) for q in qs]
    if use_ten: q2 = [normalize_temporal_expression(q) for q in q2]
    r2 = Retrieval(RETRIEVER_PATH, q2, triples, None, None)
    d2,c2 = r2.compute_similarity(n=15)
    fl = r2.get_result(d2, c2, q2, re_rank=False)

    print('[4/4] Generate...')
    results, correct = [], 0
    for i, item in enumerate(tqdm(qs, desc='Generate', leave=False)):
        q     = get_q(item)
        facts = (fl[i].get('fact') or [])[:3]
        gold  = get_gold(qs_raw[i])
        pred  = gen_answer(
            f'Answer the question using the facts. Return only the answer.\n'
            f'Facts: {facts}\nQuestion: {q}\nAnswer:'
        )
        ok = hit(pred, gold)
        if ok: correct += 1
        results.append({
            'question':  get_q(qs_raw[i]),
            'q_norm':    q_norm[i] if use_ten else '',
            'gold':      str(gold),
            'predicted': pred,
            'correct':   ok,
        })

    em = correct/len(results)*100
    print(f'\n✅ {label}: {correct}/{len(results)} = {em:.2f}%')
    return results, em

print('✅ Pipeline ready')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 5 — LẦN 1: BASELINE (không TEN)
# ═══════════════════════════════════════════════════════════

res_before, em_before = run_pipeline(
    questions, triple_list,
    use_ten=False, label='TimeR4 gốc'
)
with open('/kaggle/working/before.json', 'w', encoding='utf-8') as f:
    json.dump(res_before, f, ensure_ascii=False, indent=2)
print(f'✅ Saved before.json  |  {em_before:.2f}%')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 6 — LẦN 2: TimeR4 + TEN
# ═══════════════════════════════════════════════════════════

res_after, em_after = run_pipeline(
    questions, triple_list,
    use_ten=True, label='TimeR4 + TEN'
)
with open('/kaggle/working/after.json', 'w', encoding='utf-8') as f:
    json.dump(res_after, f, ensure_ascii=False, indent=2)
print(f'✅ Saved after.json  |  {em_after:.2f}%')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 7 — BẢNG KẾT QUẢ
# ═══════════════════════════════════════════════════════════

diff      = em_after - em_before
improved  = sum(1 for b,a in zip(res_before,res_after) if a['correct'] and not b['correct'])
degraded  = sum(1 for b,a in zip(res_before,res_after) if b['correct'] and not a['correct'])
unchanged = len(res_before) - improved - degraded
n_norm    = sum(1 for r in res_after if r['q_norm'] and r['q_norm'] != r['question'].lower())

print('═'*65)
print('  BẢNG KẾT QUẢ — BÁO CÁO VỚI THẦY')
print('═'*65)
print(f'  {"Model":<46} {"Hit@1":>8} {"Cải thiện":>9}')
print('  '+'-'*62)
print(f'  {"TimeR4 original (tiếng Anh, paper gốc)":<46} {"42.1%":>8} {"—":>9}')
print('  '+'-'*62)
print(f'  {"TimeR4 baseline (VI, chưa TEN)":<46} {em_before:>7.2f}% {"—":>9}')
ds = f'+{diff:.2f} ↑' if diff > 0 else f'{diff:.2f}'
print(f'  {"TimeR4 + TEN (VI, sau normalize)":<46} {em_after:>7.2f}% {ds:>9}')
print('═'*65)
print(f'\n  Số câu (CÓ temporal):             {len(res_before)}')
print(f'  TEN normalize được:               {n_norm}/{len(res_before)} ({n_norm/len(res_before)*100:.1f}%)')
print(f'  Cải thiện (sai→đúng):             {improved}')
print(f'  Ảnh hưởng (đúng→sai):             {degraded}')
print(f'  Không đổi:                        {unchanged}')

# Ví dụ cải thiện
ex = [(b,a) for b,a in zip(res_before,res_after) if a['correct'] and not b['correct']]
if ex:
    print('\n=== VÍ DỤ CÂU ĐƯỢC CẢI THIỆN ===')
    for b, a in ex[:3]:
        print(f'  Q:     {b["question"]}')
        if a['q_norm'] and a['q_norm'] != b['question'].lower():
            print(f'  →      {a["q_norm"]}')
        print(f'  Gold:  {str(b["gold"])[:80]}')
        print(f'  Trước: {b["predicted"][:60]}  ❌')
        print(f'  Sau:   {a["predicted"][:60]}  ✅')
        print()

summary = {
    'n': len(res_before), 'em_paper': 42.1,
    'em_before': round(em_before,2), 'em_after': round(em_after,2),
    'diff': round(diff,2),
    'coverage': f'{n_norm}/{len(res_before)} ({n_norm/len(res_before)*100:.1f}%)',
    'improved': improved, 'degraded': degraded, 'unchanged': unchanged,
}
with open('/kaggle/working/summary_final.json', 'w') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)
print('\n✅ summary_final.json saved → download từ tab Output')